In [1]:
import sys

import pandas as pd

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts

from dice4el.scenario.scenario_handler import ScenarioHandler
from dice4el.scenario.scenario_model import ScenarioLSTM, train_ScenarioLSTM, validate_ScenarioLSTM

### --- Load Dataset ---

In [2]:
set_seed(seed=42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
df = pd.read_excel(
    "../../../data/bpic20_Int.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Permit OrganizationalEntity": "string",
        "case:Amount": "float32",
        "case:RequestedAmount": "float32",
        "case:OriginalAmount": "float32",
        "case:Permit RequestedBudget": "float32",
        "case:AdjustedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,case:AdjustedAmount,case:Amount,case:OriginalAmount,case:Permit OrganizationalEntity,case:Permit RequestedBudget,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,declaration 1002,2018-03-01 10:55:17,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,declaration 1002,2018-03-01 10:55:21,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,4.0
2,declaration 1002,2018-03-01 15:01:48,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Permit FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,14787.0
3,declaration 1002,2018-03-19 00:00:00,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Start trip,STAFF MEMBER,EMPLOYEE,1501092.0
4,declaration 1002,2018-03-23 00:00:00,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,End trip,STAFF MEMBER,EMPLOYEE,345600.0
5,declaration 1002,2018-03-27 16:15:02,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,404102.0
6,declaration 1002,2018-04-03 17:07:56,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,607974.0
7,declaration 1002,2018-04-05 09:45:53,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,146277.0
8,declaration 1002,2018-04-05 17:25:23,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Request Payment,SYSTEM,UNDEFINED,27570.0
9,declaration 1002,2018-04-09 17:30:58,361.392242,361.392242,361.392242,organizational unit 65460,1273.252075,361.392242,Payment Handled,SYSTEM,UNDEFINED,345935.0


In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:AdjustedAmount', 'case:Amount', 'case:OriginalAmount', 'case:Permit OrganizationalEntity', 'case:Permit RequestedBudget', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [19.00, 518400.00]                       112149.0000 quantile_derived    
case:Amount                    continuous     case     yes    [28.52, 1883.08]                         375.8531   quantile_derived    
case:RequestedAmount           continuous     case    

In [7]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

### --- Scenario Model ---

In [8]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [9]:
scenario_df = scenario_handler.generate_scenario_df(
    df=df,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
    n_scenarios_per_length=3
)

In [10]:
scenario_df.head()

,case:concept:name,time_index,fake,case:AdjustedAmount,case:Amount,case:OriginalAmount,case:Permit OrganizationalEntity,case:Permit RequestedBudget,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,f_2125,0,True,1597.179128,1731.042107,737.777272,organizational unit 65475,1212.8878,459.602011,Start trip,STAFF MEMBER,EMPLOYEE,2657.737601
1,f_2125,1,True,1597.179128,1731.042107,737.777272,organizational unit 65475,1212.8878,459.602011,Permit FINAL_APPROVED by SUPERVISOR,SYSTEM,BUDGET OWNER,7557.780651
2,f_2125,2,True,1597.179128,1731.042107,737.777272,organizational unit 65475,1212.8878,459.602011,Declaration REJECTED by BUDGET OWNER,SYSTEM,SUPERVISOR,1417.826612
3,f_2125,3,True,1597.179128,1731.042107,737.777272,organizational unit 65475,1212.8878,459.602011,End trip,STAFF MEMBER,DIRECTOR,22.594810
4,f_2125,4,True,1597.179128,1731.042107,737.777272,organizational unit 65475,1212.8878,459.602011,Declaration SAVED by EMPLOYEE,STAFF MEMBER,PRE_APPROVER,24453.097914


In [11]:
case_ids = scenario_df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = scenario_df[scenario_df["case:concept:name"].isin(train_cases)].copy()
val_df   = scenario_df[scenario_df["case:concept:name"].isin(val_cases)].copy()

In [12]:
train_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=train_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [13]:
val_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=val_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [15]:
criterion = torch.nn.BCEWithLogitsLoss()

In [16]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Int-scenario_model_output.txt")

Epoch 020/100 | Train Loss: 0.0002 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.0000 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.0000 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.0000 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.0000 | LR: 1.00e-06
Time taken for scenario model (training): 2165.120309 seconds
Time taken for scenario model (validation): 2.163389 seconds
Val loss: {'loss': 0.0003820995396960279, 'accuracy': 0.999947966351574, 'f1_macro': 0.999929461872264, 'f1_weighted': 0.9999479669782072}


In [17]:
embedding_metadata = scenario_handler.get_scenario_embedding_metadata()

scenario_model = ScenarioLSTM(
    categorical_info=embedding_metadata["categorical_info"],
    n_continuous=embedding_metadata["n_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ScenarioLSTM(
    model=scenario_model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

scenario_model.save()

In [18]:
# scenario_model = ScenarioLSTM.load()

In [19]:
val_loss = validate_ScenarioLSTM(
    model=scenario_model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [21]:
sys.stdout = original_stdout
log_file.close()